# Rocket CSV Log Data Analysis
This notebook aims to analyze log data recorded by previous ARIS launches and drop tests.

## Analysis of CSV data

This section analyses the individual CSV files and visualizes the information provided.

In [ ]:
from pathlib import Path
import pandas as pd
from matplotlib import pyplot as plt
from scipy.spatial.transform import Rotation


# FOR Aris_Cargoflight_1_Logs
# BASE_PATH = Path("../../external/ARIS-flight-logs/flight-computer-2026-03-19T13-56-cargo-flight")
# start = pd.Timestamp("2026-03-19 15:00:00")
# end = pd.Timestamp("2026-03-19 18:00:00")

# FOR Hermes_Droptest_1_Logs
# BASE_PATH = Path("../../external/ARIS-flight-logs/Hermes_Droptest_1_Logs")
# start = pd.Timestamp("2025-05-15 12:20:00")
# end = pd.Timestamp("2025-05-15 12:52:00")

# FOR Hermes_Droptest_2_Logs
BASE_PATH = Path("../../external/ARIS-flight-logs/Hermes_Droptest_2_Logs")
start = pd.Timestamp("2025-06-11 13:00:00")
end = pd.Timestamp("2025-06-11 14:00:00")


ACCELERATION_DATA = BASE_PATH / Path("timeseries-acceleration.csv")
CURRENT_STATE_DATA = BASE_PATH / Path("timeseries-current_state.csv")
DEPLOYMENT_DATA = BASE_PATH / Path("/timeseries-deployment.csv")  # not sure what "deployment" is
ENVIRONMENT_DATA = BASE_PATH / Path("timeseries-environment.csv")
LOCATION_DATA = BASE_PATH / Path("timeseries-location.csv")
ORIENTATION_DATA = BASE_PATH / Path("timeseries-orientation.csv")
VELOCITY_DATA = BASE_PATH / Path("timeseries-velocity.csv")


In [ ]:
def load_csv_dataframe(path: Path, mask: bool = False) -> pd.DataFrame:
    df = pd.read_csv(path)
    print(f"loaded {len(df):,} lines")

    df["timestamp"] = pd.to_datetime(df["timestamp [unix ms]"], unit="ms")
    df = df.sort_values("timestamp").reset_index(drop=True)

    if mask and start is not None and end is not None:
        mask = (df["timestamp"] >= start) & (df["timestamp"] <= end)
        return df[mask].copy().reset_index(drop=True)

    return df

### Rocket State

This section visualizes the state and time spent per state.

In [ ]:
state_df = load_csv_dataframe(CURRENT_STATE_DATA, True)
state_df.head(n=20)

In [ ]:

states = state_df["state [string]"].unique()
state_map = {s: i for i, s in enumerate(states)}

state_df["state_num"] = state_df["state [string]"].map(state_map)

fig, ax = plt.subplots(figsize=(14, 6))
ax.step(state_df["timestamp"], state_df["state_num"], where="post")

ax.set_yticks(range(len(states)))
ax.set_yticklabels(states)
ax.set_xlabel("Time")
ax.set_title("Flight Computer State Transitions")
plt.tight_layout()
plt.show()

### Rocket Location

This section visualizes the location data provided.

In [ ]:
import numpy as np
import plotly.express as px

location_df = load_csv_dataframe(LOCATION_DATA, True)
state_df = load_csv_dataframe(CURRENT_STATE_DATA, True)

state_times = state_df[["timestamp", "state [string]"]].copy()
location_df = location_df.sort_values("timestamp").dropna(subset=["timestamp"]).reset_index(drop=True)

lat0 = location_df["location latitude [°]"].iloc[0]
lon0 = location_df["location longitude [°]"].iloc[0]

location_df["x_m"] = (location_df["location latitude [°]"] - lat0) * 111_320
location_df["y_m"] = (location_df["location longitude [°]"] - lon0) * 111_320 * np.cos(np.radians(lat0))

location_df = pd.merge_asof(
    location_df,
    state_times,
    on="timestamp",
    direction="backward",
)

fig = px.line_3d(
    location_df,
    x="x_m", y="y_m",
    z="location altitude [m above sea level]",
    color="state [string]",
    width=1200,
    height=800,
)
fig.show()

## Rocket Velocity, Acceleration, Orientation and Environment

This section visualizes the velocity, acceleration, orientation and environment over time

In [ ]:
velocity_df = load_csv_dataframe(VELOCITY_DATA, True)
acceleration_df = load_csv_dataframe(ACCELERATION_DATA, True)
environment_df = load_csv_dataframe(ENVIRONMENT_DATA, True)
orientation_df = load_csv_dataframe(ORIENTATION_DATA, True)

In [ ]:
velocity_df["total_velocity [m/s]"] = np.sqrt(
    velocity_df["velocity north [m/s]"] ** 2
    + velocity_df["velocity east [m/s]"] ** 2
    + velocity_df["velocity down [m/s]"] ** 2
)

fig, ax = plt.subplots(figsize=(14, 5))

for col, label in [
    ("velocity north [m/s]", "North"),
    ("velocity east [m/s]", "East"),
    ("velocity down [m/s]", "Down"),
    ("total_velocity [m/s]", "Total"),
]:
    ax.plot(velocity_df["timestamp"], velocity_df[col], label=label)

for _, row in state_df.iterrows():
    ax.axvline(row["timestamp"], color="gray", linestyle="--", linewidth=0.7, alpha=0.8)
    # ax.text(row["timestamp"], ax.get_ylim()[1], row["state [string]"], rotation=90, fontsize=6, va="top", ha="right", alpha=0.7)

ax.set_xlabel("Time")
ax.set_ylabel("Velocity [m/s]")
ax.set_title("Rocket Velocity over Time (NED + Total)")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
acceleration_df["total_acceleration [m/s^2]"] = np.sqrt(
    acceleration_df["acceleration north [m/s^2]"] ** 2
    + acceleration_df["acceleration east [m/s^2]"] ** 2
    + acceleration_df["acceleration down [m/s^2]"] ** 2
)
acceleration_df["g_load"] = acceleration_df["total_acceleration [m/s^2]"] / 9.81

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

for col, label in [
    ("acceleration north [m/s^2]", "North"),
    ("acceleration east [m/s^2]", "East"),
    ("acceleration down [m/s^2]", "Down"),
    ("total_acceleration [m/s^2]", "Total"),
]:
    ax1.plot(acceleration_df["timestamp"], acceleration_df[col], label=label)

ax2.plot(acceleration_df["timestamp"], acceleration_df["g_load"], color="tab:red", label="G-load")

for ax in (ax1, ax2):
    for _, row in state_df.iterrows():
        ax.axvline(row["timestamp"], color="gray", linestyle="--", linewidth=0.7, alpha=0.8)
        # ax.text(row["timestamp"], ax.get_ylim()[1], row["state [string]"], rotation=90, fontsize=6, va="top", ha="right", alpha=0.7)

ax1.set_ylabel("Acceleration [m/s²]")
ax1.set_title("Rocket Acceleration over Time (NED + Total)")
ax1.legend()

ax2.set_xlabel("Time")
ax2.set_ylabel("G-load [g]")
ax2.set_title("Rocket G-load over Time")
ax2.legend()

plt.tight_layout()
plt.show()

In [ ]:
from scipy.spatial.transform import Rotation

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8))

for col, label in [
    ("orientation W", "W"),
    ("orientation X", "X"),
    ("orientation Y", "Y"),
    ("orientation Z", "Z"),
]:
    ax1.plot(orientation_df["timestamp"], orientation_df[col], label=label)

quaternions = orientation_df[["orientation W", "orientation X", "orientation Y", "orientation Z"]].to_numpy()
euler_angles = Rotation.from_quat(quaternions, scalar_first=True).as_euler("ZXY", degrees=True)

ax2.plot(orientation_df["timestamp"], euler_angles[:, 0], label="Yaw")
ax2.plot(orientation_df["timestamp"], euler_angles[:, 1], label="Pitch")
ax2.plot(orientation_df["timestamp"], euler_angles[:, 2], label="Roll")

ax1.set_xlabel("Time")
ax1.set_ylabel("Orientation [Quaternions]")
ax1.set_title("Rocket Orientation over Time [Quaternions]")

ax2.set_xlabel("Time")
ax2.set_ylabel("Angle [°]")
ax2.set_title("Euler Angles (ZXY)")

plt.tight_layout()
plt.legend()
plt.show()


In [ ]:

quaternion = [1, 1, 1, 1]
rotation = Rotation.from_quat(quaternion)
rotation.as_euler("xyz", degrees=True)


In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(14, 9), sharex=True)

ax1.plot(environment_df["timestamp"], environment_df["temperature [°C]"], color="tab:red")
ax1.set_ylabel("Temperature [°C]")
ax1.set_title("Rocket Environment Sensors over Time")

ax2.plot(environment_df["timestamp"], environment_df["pressure [hPa]"], color="tab:blue")
ax2.set_ylabel("Pressure [hPa]")

ax3.plot(environment_df["timestamp"], environment_df["humidity [%]"], color="tab:green")
ax3.set_ylabel("Humidity [%]")
ax3.set_xlabel("Time")

for ax in (ax1, ax2, ax3):
    for _, row in state_df.iterrows():
        ax.axvline(row["timestamp"], color="gray", linestyle="--", linewidth=0.7, alpha=0.8)
        ax.text(row["timestamp"], ax.get_ylim()[1], row["state [string]"], rotation=90, fontsize=6, va="top",
                ha="right", alpha=0.7)

plt.tight_layout()
plt.show()

## CSV File Dimensions & Message Timing Analysis

This section compares the dimensions of all CSV files and analyzes the timestamp intervals to understand message rates and their consistency.

In [ ]:
csv_files = {
    "acceleration": ACCELERATION_DATA,
    "current_state": CURRENT_STATE_DATA,
    "environment": ENVIRONMENT_DATA,
    "location": LOCATION_DATA,
    "orientation": ORIENTATION_DATA,
    "velocity": VELOCITY_DATA,
}

dataframes = {}
dim_rows = []

for name, path in csv_files.items():
    if not path.exists():
        print(f"  SKIP {name}: file not found at {path}")
        continue
    df = pd.read_csv(path)
    dataframes[name] = df

    ts_col = "timestamp [unix ms]"
    nan_ts = int(df[ts_col].isna().sum()) if ts_col in df.columns else 0
    total_nans = int(df.isna().sum().sum())

    dim_rows.append({
        "file": name,
        "rows": len(df),
        "columns": len(df.columns),
        "NaN timestamps": nan_ts,
        "total NaNs": total_nans,
        "column_names": ", ".join(df.columns.tolist()),
    })

dim_df = pd.DataFrame(dim_rows).set_index("file")
print("=== CSV File Dimensions ===\n")
print(dim_df[["rows", "columns", "NaN timestamps", "total NaNs"]].to_string())
print(f"\nAll files share the same row count: {dim_df['rows'].nunique() == 1}")

In [ ]:
timing_rows = []

for name, df in dataframes.items():
    ts_col = "timestamp [unix ms]"
    if ts_col not in df.columns:
        continue

    ts = df[ts_col].dropna().sort_values().values
    deltas_ms = np.diff(ts).astype(float)

    if len(deltas_ms) == 0:
        continue

    timing_rows.append({
        "file": name,
        "messages": len(ts),
        "dropped NaN": len(df) - len(ts),
        "duration [s]": (ts[-1] - ts[0]) / 1000.0,
        "mean dt [ms]": np.mean(deltas_ms),
        "median dt [ms]": np.median(deltas_ms),
        "std dt [ms]": np.std(deltas_ms),
        "min dt [ms]": np.min(deltas_ms),
        "max dt [ms]": np.max(deltas_ms),
        "avg rate [Hz]": 1000.0 / np.mean(deltas_ms) if np.mean(deltas_ms) > 0 else np.inf,
    })

timing_df = pd.DataFrame(timing_rows).set_index("file")
print("=== Message Timing Statistics (full file, NaN timestamps dropped) ===\n")
print(timing_df.to_string(float_format=lambda x: f"{x:.2f}"))